#  Algoritmos de Búsqueda Desinformada

Implementación de 5 algoritmos de búsqueda fundamentales en IA:
- **BFS** (Búsqueda en Amplitud)
- **DFS** (Búsqueda en Profundidad)
- **DLS** (Profundidad Limitada)
- **IDS** (Profundización Iterativa)
- **UCS** (Costo Uniforme)

Con casos de prueba, visualización de resultados y análisis comparativo.

## 1. Importes y Configuración

In [1]:
from collections import deque
import heapq
import time
from typing import List, Tuple, Dict, Set, Optional

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


## 2. Clase Grafo - Estructura de Datos

In [2]:
class Grafo:
    """Clase para representar un grafo con costos opcionales."""
    
    def __init__(self, nombre: str = "Grafo"):
        """Inicializar el grafo."""
        self.nombre = nombre
        self.adyacencia: Dict[str, List[str]] = {}
        self.costos: Dict[Tuple[str, str], float] = {}
    
    def agregar_arista(self, origen: str, destino: str, costo: float = 1, dirigido: bool = False):
        """Agregar una arista entre dos nodos."""
        if origen not in self.adyacencia:
            self.adyacencia[origen] = []
        if destino not in self.adyacencia:
            self.adyacencia[destino] = []
        
        if destino not in self.adyacencia[origen]:
            self.adyacencia[origen].append(destino)
        self.costos[(origen, destino)] = costo
        
        if not dirigido:
            if origen not in self.adyacencia[destino]:
                self.adyacencia[destino].append(origen)
            self.costos[(destino, origen)] = costo
    
    def obtener_nodos(self) -> List[str]:
        """Obtener lista de todos los nodos."""
        return list(self.adyacencia.keys())
    
    def __str__(self) -> str:
        """Representación en string del grafo."""
        lineas = [f"\n{'='*50}"]
        lineas.append(f"Grafo: {self.nombre}")
        lineas.append(f"Nodos: {', '.join(self.obtener_nodos())}")
        lineas.append(f"Cantidad de nodos: {len(self.adyacencia)}")
        lineas.append(f"{'='*50}\n")
        
        for nodo in sorted(self.adyacencia.keys()):
            vecinos = []
            for vecino in self.adyacencia[nodo]:
                costo = self.costos.get((nodo, vecino), 1)
                vecinos.append(f"{vecino}({costo})")
            lineas.append(f"{nodo:15} -> {', '.join(vecinos)}")
        
        return '\n'.join(lineas)

print("✓ Clase Grafo definida")

✓ Clase Grafo definida


## 3. Algoritmo 1: BFS (Búsqueda en Amplitud)

In [3]:
def busqueda_amplitud(grafo_dict: Dict, inicio: str, objetivo: str) -> Tuple[Optional[List[str]], Set[str], int]:
    """
    Búsqueda Primero en Amplitud (BFS)
    - Estructura: Cola (FIFO)
    - Óptima: Sí (en número de aristas)
    - Completa: Sí
    """
    cola = deque([inicio])
    visitados = {inicio}
    padres = {inicio: None}
    nodos_explorados = set()
    
    while cola:
        nodo_actual = cola.popleft()
        nodos_explorados.add(nodo_actual)
        
        if nodo_actual == objetivo:
            # Reconstruir camino
            camino = []
            nodo = objetivo
            while nodo is not None:
                camino.append(nodo)
                nodo = padres[nodo]
            camino.reverse()
            return camino, nodos_explorados, len(nodos_explorados)
        
        if nodo_actual in grafo_dict:
            for vecino in grafo_dict[nodo_actual]:
                if vecino not in visitados:
                    visitados.add(vecino)
                    padres[vecino] = nodo_actual
                    cola.append(vecino)
    
    return None, nodos_explorados, len(nodos_explorados)

print("✓ BFS implementado")

✓ BFS implementado


## 4. Algoritmo 2: DFS (Búsqueda en Profundidad)

In [4]:
def busqueda_profundidad(grafo_dict: Dict, inicio: str, objetivo: str) -> Tuple[Optional[List[str]], Set[str], int]:
    """
    Búsqueda en Profundidad (DFS)
    - Estructura: Pila (LIFO)
    - Óptima: No
    - Completa: Sí (en grafos finitos)
    """
    pila = [inicio]
    visitados = {inicio}
    padres = {inicio: None}
    nodos_explorados = set()
    
    while pila:
        nodo_actual = pila.pop()
        nodos_explorados.add(nodo_actual)
        
        if nodo_actual == objetivo:
            camino = []
            nodo = objetivo
            while nodo is not None:
                camino.append(nodo)
                nodo = padres[nodo]
            camino.reverse()
            return camino, nodos_explorados, len(nodos_explorados)
        
        if nodo_actual in grafo_dict:
            for vecino in reversed(grafo_dict[nodo_actual]):
                if vecino not in visitados:
                    visitados.add(vecino)
                    padres[vecino] = nodo_actual
                    pila.append(vecino)
    
    return None, nodos_explorados, len(nodos_explorados)

print("✓ DFS implementado")

✓ DFS implementado


## 5. Algoritmo 3: DLS (Búsqueda en Profundidad Limitada)

In [5]:
def busqueda_profundidad_limitada(grafo_dict: Dict, inicio: str, objetivo: str, 
                                   limite: int) -> Tuple[Optional[List[str]], Set[str], int]:
    """
    Búsqueda en Profundidad Limitada (DLS)
    - Estructura: Pila con límite de profundidad
    - Óptima: No
    - Completa: Parcial (si la solución está dentro del límite)
    """
    nodos_explorados = set()
    
    def dls_recursivo(nodo: str, profundidad: int, padres: Dict) -> Optional[List[str]]:
        nodos_explorados.add(nodo)
        
        if nodo == objetivo:
            camino = []
            n = objetivo
            while n is not None:
                camino.append(n)
                n = padres[n]
            camino.reverse()
            return camino
        
        if profundidad == 0:
            return None
        
        if nodo in grafo_dict:
            for vecino in grafo_dict[nodo]:
                if vecino not in padres:
                    padres[vecino] = nodo
                    resultado = dls_recursivo(vecino, profundidad - 1, padres)
                    if resultado is not None:
                        return resultado
                    del padres[vecino]
        
        return None
    
    padres = {inicio: None}
    camino = dls_recursivo(inicio, limite, padres)
    
    return camino, nodos_explorados, len(nodos_explorados)

print("✓ DLS implementado")

✓ DLS implementado


## 6. Algoritmo 4: IDS (Profundización Iterativa)

In [6]:
def busqueda_profundizacion_iterativa(grafo_dict: Dict, inicio: str, objetivo: str, 
                                       max_profundidad: int = None) -> Tuple[Optional[List[str]], Set[str], int]:
    """
    Búsqueda de Profundización Iterativa (IDS)
    - Estructura: DLS repetido con límites incrementales
    - Óptima: Sí
    - Completa: Sí
    """
    todos_nodos_explorados = set()
    
    def dls_recursivo(nodo: str, profundidad: int, visitados: Set[str], padres: Dict) -> Optional[List[str]]:
        todos_nodos_explorados.add(nodo)
        visitados.add(nodo)
        
        if nodo == objetivo:
            camino = []
            n = objetivo
            while n is not None:
                camino.append(n)
                n = padres[n]
            camino.reverse()
            return camino
        
        if profundidad == 0:
            return None
        
        if nodo in grafo_dict:
            for vecino in grafo_dict[nodo]:
                if vecino not in visitados:
                    padres[vecino] = nodo
                    resultado = dls_recursivo(vecino, profundidad - 1, visitados, padres)
                    if resultado is not None:
                        return resultado
        
        return None
    
    if max_profundidad is None:
        max_profundidad = len(grafo_dict)
    
    for limite_actual in range(max_profundidad + 1):
        visitados = set()
        padres = {inicio: None}
        resultado = dls_recursivo(inicio, limite_actual, visitados, padres)
        
        if resultado is not None:
            return resultado, todos_nodos_explorados, len(todos_nodos_explorados)
    
    return None, todos_nodos_explorados, len(todos_nodos_explorados)

print("✓ IDS implementado")

✓ IDS implementado


## 7. Algoritmo 5: UCS (Búsqueda de Costo Uniforme)

In [7]:
def busqueda_costo_uniforme(grafo_dict: Dict, costos: Dict, inicio: str, 
                             objetivo: str) -> Tuple[Optional[List[str]], Set[str], int, Optional[float]]:
    """
    Búsqueda de Costo Uniforme (UCS)
    - Estructura: Cola de prioridad
    - Óptima: Sí (en costo total)
    - Completa: Sí
    """
    heap = [(0, inicio)]
    costo_g = {inicio: 0}
    padres = {inicio: None}
    nodos_explorados = set()
    visitados = set()
    
    while heap:
        costo_actual, nodo_actual = heapq.heappop(heap)
        
        if nodo_actual in visitados:
            continue
        
        visitados.add(nodo_actual)
        nodos_explorados.add(nodo_actual)
        
        if nodo_actual == objetivo:
            camino = []
            nodo = objetivo
            while nodo is not None:
                camino.append(nodo)
                nodo = padres[nodo]
            camino.reverse()
            
            costo_final = costo_g[objetivo]
            return camino, nodos_explorados, len(nodos_explorados), costo_final
        
        if nodo_actual in grafo_dict:
            for vecino in grafo_dict[nodo_actual]:
                if vecino not in visitados:
                    arista = (nodo_actual, vecino)
                    costo_arista = costos.get(arista, 1)
                    
                    nuevo_costo = costo_actual + costo_arista
                    
                    if vecino not in costo_g or nuevo_costo < costo_g[vecino]:
                        costo_g[vecino] = nuevo_costo
                        padres[vecino] = nodo_actual
                        heapq.heappush(heap, (nuevo_costo, vecino))
    
    return None, nodos_explorados, len(nodos_explorados), None

print("✓ UCS implementado")

✓ UCS implementado


## 8. Casos de Prueba

### Caso 1: Grafo Simple

In [8]:
def obtener_grafo_simple():
    """Grafo simple para pruebas rápidas."""
    grafo = Grafo("Grafo Simple")
    
    aristas = [
        ("A", "B", 1),
        ("A", "C", 2),
        ("B", "D", 1),
        ("B", "E", 3),
        ("C", "E", 1),
        ("D", "F", 2),
        ("E", "F", 1),
    ]
    
    for origen, destino, costo in aristas:
        grafo.agregar_arista(origen, destino, costo, dirigido=False)
    
    return grafo, "A", "F"

grafo1, inicio1, objetivo1 = obtener_grafo_simple()
print(grafo1)
print(f"Buscando ruta: {inicio1} → {objetivo1}")


Grafo: Grafo Simple
Nodos: A, B, C, D, E, F
Cantidad de nodos: 6

A               -> B(1), C(2)
B               -> A(1), D(1), E(3)
C               -> A(2), E(1)
D               -> B(1), F(2)
E               -> B(3), C(1), F(1)
F               -> D(2), E(1)
Buscando ruta: A → F


### Caso 2: Ruta de Ciudades

In [9]:
def obtener_grafo_ciudades():
    """Grafo de ciudades conectadas."""
    grafo = Grafo("Ruta de Ciudades")
    
    conexiones = [
        ("Madrid", "Barcelona", 630),
        ("Madrid", "Valencia", 310),
        ("Barcelona", "Tarragona", 180),
        ("Valencia", "Tarragona", 260),
        ("Valencia", "Alicante", 250),
    ]
    
    for origen, destino, distancia in conexiones:
        grafo.agregar_arista(origen, destino, distancia, dirigido=False)
    
    return grafo, "Madrid", "Alicante"

grafo2, inicio2, objetivo2 = obtener_grafo_ciudades()
print(grafo2)
print(f"Buscando ruta: {inicio2} → {objetivo2}")


Grafo: Ruta de Ciudades
Nodos: Madrid, Barcelona, Valencia, Tarragona, Alicante
Cantidad de nodos: 5

Alicante        -> Valencia(250)
Barcelona       -> Madrid(630), Tarragona(180)
Madrid          -> Barcelona(630), Valencia(310)
Tarragona       -> Barcelona(180), Valencia(260)
Valencia        -> Madrid(310), Tarragona(260), Alicante(250)
Buscando ruta: Madrid → Alicante


### Caso 3: Mapa con Costos

In [10]:
def obtener_mapa_costos():
    """Grafo con costos variables."""
    grafo = Grafo("Mapa con Costos")
    
    aristas = [
        ("A", "B", 2),
        ("A", "C", 3),
        ("B", "D", 4),
        ("B", "E", 1),
        ("C", "F", 2),
        ("C", "G", 5),
        ("D", "H", 1),
        ("E", "H", 2),
        ("F", "I", 3),
    ]
    
    for origen, destino, costo in aristas:
        grafo.agregar_arista(origen, destino, costo, dirigido=False)
    
    return grafo, "A", "H"

grafo3, inicio3, objetivo3 = obtener_mapa_costos()
print(grafo3)
print(f"Buscando camino: {inicio3} → {objetivo3}")


Grafo: Mapa con Costos
Nodos: A, B, C, D, E, F, G, H, I
Cantidad de nodos: 9

A               -> B(2), C(3)
B               -> A(2), D(4), E(1)
C               -> A(3), F(2), G(5)
D               -> B(4), H(1)
E               -> B(1), H(2)
F               -> C(2), I(3)
G               -> C(5)
H               -> D(1), E(2)
I               -> F(3)
Buscando camino: A → H


### Caso 4: Laberinto

In [11]:
def obtener_laberinto():
    """Grafo que representa un laberinto 3x4.
    
    Matriz:
    1 1 0 1
    0 1 1 1
    1 1 0 1
    """
    grafo = Grafo("Laberinto 3x4")
    
    laberinto = [
        [1, 1, 0, 1],
        [0, 1, 1, 1],
        [1, 1, 0, 1]
    ]
    
    # Crear nodos
    nodos_validos = []
    for i in range(len(laberinto)):
        for j in range(len(laberinto[i])):
            if laberinto[i][j] == 1:
                nodo = f"({i},{j})"
                nodos_validos.append((i, j, nodo))
                grafo.adyacencia[nodo] = []
    
    # Conectar nodos adyacentes
    direcciones = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    for i, j, nodo_actual in nodos_validos:
        for di, dj in direcciones:
            ni, nj = i + di, j + dj
            
            if (0 <= ni < len(laberinto) and 
                0 <= nj < len(laberinto[0]) and 
                laberinto[ni][nj] == 1):
                
                nodo_vecino = f"({ni},{nj})"
                if nodo_vecino not in grafo.adyacencia[nodo_actual]:
                    grafo.agregar_arista(nodo_actual, nodo_vecino, costo=1, dirigido=False)
    
    return grafo, "(0,0)", "(2,3)"

grafo4, inicio4, objetivo4 = obtener_laberinto()
print(grafo4)
print(f"Buscando salida: {inicio4} → {objetivo4}")


Grafo: Laberinto 3x4
Nodos: (0,0), (0,1), (0,3), (1,1), (1,2), (1,3), (2,0), (2,1), (2,3)
Cantidad de nodos: 9

(0,0)           -> (0,1)(1)
(0,1)           -> (0,0)(1), (1,1)(1)
(0,3)           -> (1,3)(1)
(1,1)           -> (0,1)(1), (2,1)(1), (1,2)(1)
(1,2)           -> (1,1)(1), (1,3)(1)
(1,3)           -> (0,3)(1), (1,2)(1), (2,3)(1)
(2,0)           -> (2,1)(1)
(2,1)           -> (1,1)(1), (2,0)(1)
(2,3)           -> (1,3)(1)
Buscando salida: (0,0) → (2,3)


## 9. Función de Comparación

In [12]:
def ejecutar_comparacion_completa(grafo, inicio: str, objetivo: str):
    """Ejecutar todos los algoritmos y mostrar comparativa."""
    
    print(f"\n{'='*80}")
    print(f"COMPARACIÓN DE ALGORITMOS: {grafo.nombre}")
    print(f"Buscando: {inicio} → {objetivo}")
    print(f"{'='*80}\n")
    
    resultados = {}
    
    # BFS
    inicio_t = time.time()
    camino, explorados, num_exp = busqueda_amplitud(grafo.adyacencia, inicio, objetivo)
    tiempo = time.time() - inicio_t
    print(f"✓ BFS: Camino={'Encontrado' if camino else 'NO encontrado'} | Nodos={num_exp} | Tiempo={tiempo*1000:.4f}ms")
    if camino:
        print(f"  Ruta: {' → '.join(camino)}")
    resultados['BFS'] = {'camino': camino, 'nodos': num_exp, 'costo': None, 'tiempo': tiempo}
    
    # DFS
    inicio_t = time.time()
    camino, explorados, num_exp = busqueda_profundidad(grafo.adyacencia, inicio, objetivo)
    tiempo = time.time() - inicio_t
    print(f"✓ DFS: Camino={'Encontrado' if camino else 'NO encontrado'} | Nodos={num_exp} | Tiempo={tiempo*1000:.4f}ms")
    if camino:
        print(f"  Ruta: {' → '.join(camino)}")
    resultados['DFS'] = {'camino': camino, 'nodos': num_exp, 'costo': None, 'tiempo': tiempo}
    
    # DLS
    limite = len(grafo.adyacencia) - 1
    inicio_t = time.time()
    camino, explorados, num_exp = busqueda_profundidad_limitada(grafo.adyacencia, inicio, objetivo, limite)
    tiempo = time.time() - inicio_t
    print(f"✓ DLS (límite={limite}): Camino={'Encontrado' if camino else 'NO encontrado'} | Nodos={num_exp} | Tiempo={tiempo*1000:.4f}ms")
    if camino:
        print(f"  Ruta: {' → '.join(camino)}")
    resultados['DLS'] = {'camino': camino, 'nodos': num_exp, 'costo': None, 'tiempo': tiempo}
    
    # IDS
    inicio_t = time.time()
    camino, explorados, num_exp = busqueda_profundizacion_iterativa(grafo.adyacencia, inicio, objetivo)
    tiempo = time.time() - inicio_t
    print(f"✓ IDS: Camino={'Encontrado' if camino else 'NO encontrado'} | Nodos={num_exp} | Tiempo={tiempo*1000:.4f}ms")
    if camino:
        print(f"  Ruta: {' → '.join(camino)}")
    resultados['IDS'] = {'camino': camino, 'nodos': num_exp, 'costo': None, 'tiempo': tiempo}
    
    # UCS
    inicio_t = time.time()
    camino, explorados, num_exp, costo = busqueda_costo_uniforme(grafo.adyacencia, grafo.costos, inicio, objetivo)
    tiempo = time.time() - inicio_t
    print(f"✓ UCS: Camino={'Encontrado' if camino else 'NO encontrado'} | Nodos={num_exp} | Costo={costo if costo else 'N/A'} | Tiempo={tiempo*1000:.4f}ms")
    if camino:
        print(f"  Ruta: {' → '.join(camino)}")
    resultados['UCS'] = {'camino': camino, 'nodos': num_exp, 'costo': costo, 'tiempo': tiempo}
    
    return resultados

print("✓ Función de comparación definida")

✓ Función de comparación definida


## 10. Ejecución de Pruebas

### Prueba 1: Grafo Simple

In [13]:
resultados1 = ejecutar_comparacion_completa(grafo1, inicio1, objetivo1)


COMPARACIÓN DE ALGORITMOS: Grafo Simple
Buscando: A → F

✓ BFS: Camino=Encontrado | Nodos=6 | Tiempo=0.0000ms
  Ruta: A → B → D → F
✓ DFS: Camino=Encontrado | Nodos=4 | Tiempo=0.0000ms
  Ruta: A → B → D → F
✓ DLS (límite=5): Camino=Encontrado | Nodos=4 | Tiempo=0.0000ms
  Ruta: A → B → D → F
✓ IDS: Camino=Encontrado | Nodos=6 | Tiempo=0.0000ms
  Ruta: A → B → D → F
✓ UCS: Camino=Encontrado | Nodos=6 | Costo=4 | Tiempo=0.0000ms
  Ruta: A → B → D → F


### Prueba 2: Ruta de Ciudades

In [14]:
resultados2 = ejecutar_comparacion_completa(grafo2, inicio2, objetivo2)


COMPARACIÓN DE ALGORITMOS: Ruta de Ciudades
Buscando: Madrid → Alicante

✓ BFS: Camino=Encontrado | Nodos=5 | Tiempo=0.0000ms
  Ruta: Madrid → Valencia → Alicante
✓ DFS: Camino=Encontrado | Nodos=5 | Tiempo=0.0000ms
  Ruta: Madrid → Valencia → Alicante
✓ DLS (límite=4): Camino=Encontrado | Nodos=5 | Tiempo=0.0000ms
  Ruta: Madrid → Barcelona → Tarragona → Valencia → Alicante
✓ IDS: Camino=Encontrado | Nodos=5 | Tiempo=0.0000ms
  Ruta: Madrid → Valencia → Alicante
✓ UCS: Camino=Encontrado | Nodos=3 | Costo=560 | Tiempo=0.0000ms
  Ruta: Madrid → Valencia → Alicante


### Prueba 3: Mapa con Costos

In [15]:
resultados3 = ejecutar_comparacion_completa(grafo3, inicio3, objetivo3)


COMPARACIÓN DE ALGORITMOS: Mapa con Costos
Buscando: A → H

✓ BFS: Camino=Encontrado | Nodos=8 | Tiempo=0.0000ms
  Ruta: A → B → D → H
✓ DFS: Camino=Encontrado | Nodos=4 | Tiempo=0.0000ms
  Ruta: A → B → D → H
✓ DLS (límite=8): Camino=Encontrado | Nodos=4 | Tiempo=0.0000ms
  Ruta: A → B → D → H
✓ IDS: Camino=Encontrado | Nodos=8 | Tiempo=0.0000ms
  Ruta: A → B → D → H
✓ UCS: Camino=Encontrado | Nodos=6 | Costo=5 | Tiempo=0.0000ms
  Ruta: A → B → E → H


### Prueba 4: Laberinto

In [16]:
resultados4 = ejecutar_comparacion_completa(grafo4, inicio4, objetivo4)


COMPARACIÓN DE ALGORITMOS: Laberinto 3x4
Buscando: (0,0) → (2,3)

✓ BFS: Camino=Encontrado | Nodos=9 | Tiempo=0.0000ms
  Ruta: (0,0) → (0,1) → (1,1) → (1,2) → (1,3) → (2,3)
✓ DFS: Camino=Encontrado | Nodos=9 | Tiempo=0.0000ms
  Ruta: (0,0) → (0,1) → (1,1) → (1,2) → (1,3) → (2,3)
✓ DLS (límite=8): Camino=Encontrado | Nodos=9 | Tiempo=0.0000ms
  Ruta: (0,0) → (0,1) → (1,1) → (1,2) → (1,3) → (2,3)
✓ IDS: Camino=Encontrado | Nodos=9 | Tiempo=0.0000ms
  Ruta: (0,0) → (0,1) → (1,1) → (1,2) → (1,3) → (2,3)
✓ UCS: Camino=Encontrado | Nodos=9 | Costo=5 | Tiempo=0.0000ms
  Ruta: (0,0) → (0,1) → (1,1) → (1,2) → (1,3) → (2,3)


## 11. Tabla Comparativa Global

In [17]:
import pandas as pd

# Crear tabla comparativa para Grafo Simple
data = []
for algo, res in resultados1.items():
    data.append({
        'Algoritmo': algo,
        'Camino': 'Sí' if res['camino'] else 'No',
        'Nodos Explorados': res['nodos'],
        'Costo': res['costo'] if res['costo'] is not None else 'N/A',
        'Tiempo (ms)': f"{res['tiempo']*1000:.4f}"
    })

df = pd.DataFrame(data)
print("\n📊 TABLA COMPARATIVA - Grafo Simple:\n")
print(df.to_string(index=False))


📊 TABLA COMPARATIVA - Grafo Simple:

Algoritmo Camino  Nodos Explorados Costo Tiempo (ms)
      BFS     Sí                 6   N/A      0.0000
      DFS     Sí                 4   N/A      0.0000
      DLS     Sí                 4   N/A      0.0000
      IDS     Sí                 6   N/A      0.0000
      UCS     Sí                 6     4      0.0000


## 12. Análisis y Conclusiones

In [18]:
print("""
╔════════════════════════════════════════════════════════════════════╗
║           ANÁLISIS DE ALGORITMOS DE BÚSQUEDA                        ║
╚════════════════════════════════════════════════════════════════════╝

📊 COMPARATIVA POR CARACTERÍSTICAS:

┌─────────────┬──────────────┬─────────┬───────────┬──────────┐
│ Algoritmo   │ Estructuras  │ Óptima  │ Completa  │ Memoria  │
├─────────────┼──────────────┼─────────┼───────────┼──────────┤
│ BFS         │ Cola (FIFO)  │ ✓ Sí   │ ✓ Sí      │ ↑ Alto   │
│ DFS         │ Pila (LIFO)  │ ✗ No   │ ✓ Sí      │ ↓ Bajo   │
│ DLS         │ Pila+Límite  │ ✗ No   │ Parcial   │ ↓ Bajo   │
│ IDS         │ DLS Iterado  │ ✓ Sí   │ ✓ Sí      │ ↔ Medio  │
│ UCS         │ Cola Prioridad│ ✓ Sí   │ ✓ Sí      │ ↑ Alto   │
└─────────────┴──────────────┴─────────┴───────────┴──────────┘

💡 RECOMENDACIONES DE USO:

1. BFS (Búsqueda en Amplitud)
   → Usar cuando: Necesitas el camino más corto (en pasos)
   → Ventaja: Garantiza optimalidad en número de aristas
   → Desventaja: Usa mucha memoria
   → Ideal para: Grafos pequeños y medianos

2. DFS (Búsqueda en Profundidad)
   → Usar cuando: Necesitas exploración exhaustiva
   → Ventaja: Uso eficiente de memoria
   → Desventaja: No garantiza optimalidad
   → Ideal para: Detección de ciclos, conectividad

3. DLS (Profundidad Limitada)
   → Usar cuando: Conoces una cota máxima de profundidad
   → Ventaja: Evita búsqueda infinita
   → Desventaja: No garantiza encontrar la solución
   → Ideal para: Grafos muy profundos con límite conocido

4. IDS (Profundización Iterativa)
   → Usar cuando: Necesitas optimalidad con poco espacio
   → Ventaja: Combina ventajas de BFS y DFS
   → Desventaja: Explora nodos múltiples veces
   → Ideal para: Espacios muy grandes

5. UCS (Costo Uniforme)
   → Usar cuando: Los costos de aristas son variables
   → Ventaja: Encuentra el camino de menor costo
   → Desventaja: Complejidad mayor que BFS
   → Ideal para: Grafos ponderados reales

🎯 CONCLUSIÓN:

• Para optimalidad garantizada: BFS, IDS, UCS
• Para eficiencia de memoria: DFS, DLS
• Para problemas con costos: UCS
• Para espacios muy grandes: IDS
• Para grafos pequeños: BFS
""")


╔════════════════════════════════════════════════════════════════════╗
║           ANÁLISIS DE ALGORITMOS DE BÚSQUEDA                        ║
╚════════════════════════════════════════════════════════════════════╝

📊 COMPARATIVA POR CARACTERÍSTICAS:

┌─────────────┬──────────────┬─────────┬───────────┬──────────┐
│ Algoritmo   │ Estructuras  │ Óptima  │ Completa  │ Memoria  │
├─────────────┼──────────────┼─────────┼───────────┼──────────┤
│ BFS         │ Cola (FIFO)  │ ✓ Sí   │ ✓ Sí      │ ↑ Alto   │
│ DFS         │ Pila (LIFO)  │ ✗ No   │ ✓ Sí      │ ↓ Bajo   │
│ DLS         │ Pila+Límite  │ ✗ No   │ Parcial   │ ↓ Bajo   │
│ IDS         │ DLS Iterado  │ ✓ Sí   │ ✓ Sí      │ ↔ Medio  │
│ UCS         │ Cola Prioridad│ ✓ Sí   │ ✓ Sí      │ ↑ Alto   │
└─────────────┴──────────────┴─────────┴───────────┴──────────┘

💡 RECOMENDACIONES DE USO:

1. BFS (Búsqueda en Amplitud)
   → Usar cuando: Necesitas el camino más corto (en pasos)
   → Ventaja: Garantiza optimalidad en número de aristas
  